# 2026 FIFA World Cup — Dixon-Coles (double-Poisson) variant

A companion to `wc2026_predictions.ipynb`. That notebook models the **score difference**
as Student-t and classifies win/draw/loss with a ±0.5 threshold. Here we instead model the
**two scorelines** with independent Poissons plus the **Dixon-Coles low-score correction**,
which (a) produces proper draw probabilities with no threshold, (b) adds an explicit home
advantage, and (c) yields full scoreline distributions. The teams, FIFA-ranking prior, and
in-scope training filter are identical, so the two models are directly comparable.

**Model.** For a match between home team *h* and away team *a*:

$$\lambda_h = \exp(\mu_0 + \gamma\,\mathbb{1}[\text{not neutral}] + \text{att}_h - \text{def}_a),\qquad
  \lambda_a = \exp(\mu_0 + \text{att}_a - \text{def}_h)$$
$$\text{goals}_h \sim \text{Poisson}(\lambda_h),\quad \text{goals}_a \sim \text{Poisson}(\lambda_a)$$

with the Dixon-Coles dependence factor \(\tau\) applied to the four low scores
(0-0, 1-0, 0-1, 1-1) and dependence parameter \(\rho\) (negative \(\rho\) lifts draws).
Attack and defence each get a hierarchical FIFA-ranking prior.

---

In [ ]:
# --- Environment setup (must run before the Stan model is compiled) ---
# Homebrew Python here was built against an SDK path that no longer exists, so the C++
# linker fails with "library 'c++' not found". Point it at the current SDK.
import os, sys, subprocess
if sys.platform == "darwin":
    try:
        sdk = subprocess.check_output(["xcrun", "--show-sdk-path"], text=True).strip()
        if sdk and os.path.isdir(sdk):
            os.environ["SDKROOT"]     = sdk
            os.environ["CFLAGS"]      = f"-isysroot {sdk}"
            os.environ["CPPFLAGS"]    = f"-isysroot {sdk}"
            os.environ["LDFLAGS"]     = f"-isysroot {sdk}"
            os.environ["LDSHARED"]    = f"clang -bundle -undefined dynamic_lookup -isysroot {sdk}"
            os.environ["LDCXXSHARED"] = f"clang++ -bundle -undefined dynamic_lookup -isysroot {sdk}"
            print(f"SDK configured: {sdk}")
        else:
            print("xcrun returned no valid SDK path; leaving compiler env untouched.")
    except Exception as e:
        print(f"Could not query SDK via xcrun ({e}); leaving compiler env untouched.")

## 0. Setup

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # PyStan 3 calls asyncio.run() internally; allow it inside the kernel

import stan, json, urllib.request, io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from scipy.stats import poisson, t as t_dist
import warnings; warnings.filterwarnings('ignore')

BLUE, RED, GRAY, GOLD, DARK, BG = '#1d6fa4','#c0392b','#7f8c8d','#e67e22','#2c3e50','#f8f9fa'
print("Libraries loaded.")

## 1. Teams, groups and FIFA rankings

Identical to the Student-t notebook: 12 groups of 4, and the real 3 April 2025 FIFA ranking over the 62-team model universe (48 qualifiers + 14 top-50 opponents).

In [ ]:
groups = {
    'A': ["Mexico","South Africa","South Korea","Czech Republic"],
    'B': ["Canada","Bosnia and Herzegovina","Qatar","Switzerland"],
    'C': ["Brazil","Morocco","Haiti","Scotland"],
    'D': ["United States","Paraguay","Australia","Turkey"],
    'E': ["Germany","Curaçao","Ivory Coast","Ecuador"],
    'F': ["Netherlands","Japan","Sweden","Tunisia"],
    'G': ["Belgium","Egypt","Iran","New Zealand"],
    'H': ["Spain","Cape Verde","Saudi Arabia","Uruguay"],
    'I': ["France","Senegal","Iraq","Norway"],
    'J': ["Argentina","Algeria","Austria","Jordan"],
    'K': ["Portugal","DR Congo","Uzbekistan","Colombia"],
    'L': ["England","Croatia","Ghana","Panama"],
}
qualified  = {t for ts in groups.values() for t in ts}
team_group = {t: g for g, ts in groups.items() for t in ts}

# Real FIFA/Coca-Cola Men's World Ranking, 3 April 2025 (release id14702). 62-team universe.
rankings = {
    "Argentina":1,"Spain":2,"France":3,"England":4,"Brazil":5,"Netherlands":6,"Portugal":7,"Belgium":8,"Italy":9,"Germany":10,
    "Croatia":11,"Morocco":12,"Uruguay":13,"Colombia":14,"Japan":15,"United States":16,"Mexico":17,"Iran":18,"Senegal":19,"Switzerland":20,
    "Denmark":21,"Austria":22,"South Korea":23,"Ecuador":24,"Ukraine":25,"Australia":26,"Turkey":27,"Sweden":28,"Wales":29,"Canada":30,
    "Serbia":31,"Egypt":32,"Panama":33,"Poland":34,"Algeria":36,"Hungary":37,"Norway":38,"Czech Republic":39,"Greece":40,"Ivory Coast":41,
    "Peru":42,"Nigeria":43,"Scotland":44,"Romania":45,"Slovakia":46,"Venezuela":47,"Paraguay":48,"Tunisia":49,"Cameroon":50,"Qatar":55,
    "South Africa":56,"Uzbekistan":57,"Saudi Arabia":58,"Iraq":59,"DR Congo":61,"Jordan":62,"Bosnia and Herzegovina":70,"Cape Verde":72,"Ghana":76,"Haiti":83,
    "New Zealand":86,"Curaçao":90,
}
teams       = sorted(rankings.keys())
team_idx    = {t: i+1 for i, t in enumerate(teams)}
ranks       = np.array([rankings[t] for t in teams], dtype=float)
inv_ranks   = ranks.max() - ranks + 1
prior_score = (inv_ranks - inv_ranks.mean()) / (2 * inv_ranks.std(ddof=1))
RANK_THRESHOLD = 50
in_scope = qualified | {t for t, r in rankings.items() if r <= RANK_THRESHOLD}
print(f"{len(teams)}-team universe ({len(qualified)} qualified + {len(teams)-len(qualified)} opponents)")

## 2. Training data — goals, home/away and neutral venue

The Dixon-Coles likelihood needs **both goal counts** and **which side was at home**, not just
the score difference. So unlike the Student-t notebook we keep `home_goals`, `away_goals`,
`home_idx`, `away_idx` and a `not_neutral` flag (from the dataset's own `neutral` column).
The in-scope filter (qualifier or FIFA top-50) is unchanged; 2022 WC games stay
qualifier-vs-qualifier.

In [ ]:
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
req = urllib.request.Request(url, headers={"User-Agent":"WC2026Bot/1.0"})
with urllib.request.urlopen(req) as resp:
    df_all = pd.read_csv(io.BytesIO(resp.read()))

cols = ['date','home_team','away_team','home_score','away_score','neutral']
wc22 = df_all[
    (df_all['tournament']=='FIFA World Cup') &
    (df_all['date']>='2022-11-01') & (df_all['date']<'2023-01-01') &
    (df_all['home_team'].isin(qualified)) & (df_all['away_team'].isin(qualified))
][cols].copy(); wc22['source']='wc2022'

qual26 = df_all[
    (df_all['tournament']=='FIFA World Cup qualification') &
    (df_all['date']>'2022-12-18') &
    (df_all['home_team'].isin(in_scope)) & (df_all['away_team'].isin(in_scope)) &
    (~df_all['home_score'].isna())
][cols].copy(); qual26['source']='qual2026'

matches = pd.concat([wc22, qual26], ignore_index=True)

def build_training_dc(matches, team_idx):
    rows = []
    for _, r in matches.iterrows():
        h, a = r['home_team'], r['away_team']
        if h in team_idx and a in team_idx:
            rows.append({'home_idx': team_idx[h], 'away_idx': team_idx[a],
                         'home_goals': int(r['home_score']), 'away_goals': int(r['away_score']),
                         'not_neutral': 0 if r['neutral'] else 1})
    return pd.DataFrame(rows)

training = build_training_dc(matches, team_idx)
print(f"Training games: {len(training)}  ({len(wc22)} WC2022 + {len(qual26)} qualifying)")
print(f"  home-advantage (non-neutral) games: {int(training['not_neutral'].sum())}")
print(f"  neutral-venue games:               {int((training['not_neutral']==0).sum())}")

## 3. The Dixon-Coles model

Independent Poissons for the two scorelines, with the Dixon-Coles correction $\tau$ on the
four low scores giving the joint pmf

$$P(X=x, Y=y) = \tau(x,y;\lambda_h,\lambda_a,\rho)\;\text{Pois}(x\mid\lambda_h)\,\text{Pois}(y\mid\lambda_a)$$

where $\tau$ is 1 except $\tau(0,0)=1-\lambda_h\lambda_a\rho$, $\tau(1,0)=1+\lambda_a\rho$,
$\tau(0,1)=1+\lambda_h\rho$, $\tau(1,1)=1-\rho$. Attack and defence carry the FIFA-ranking prior:
$\text{att}_i\sim\mathcal N(b_\text{att}s_i,\sigma_\text{att})$,
$\text{def}_i\sim\mathcal N(b_\text{def}s_i,\sigma_\text{def})$ (non-centred). A couple of
"Rejecting initial value" warnings at the start are benign — random inits can momentarily make
$\tau\le 0$; Stan re-initialises and samples normally.

In [ ]:
dc_code = """
functions {
  real dc_log_tau(int x, int y, real lh, real la, real rho) {
    if (x == 0 && y == 0) return log(1 - lh * la * rho);
    if (x == 0 && y == 1) return log(1 + lh * rho);
    if (x == 1 && y == 0) return log(1 + la * rho);
    if (x == 1 && y == 1) return log(1 - rho);
    return 0;
  }
}
data {
  int<lower=0> num_teams;
  int<lower=0> num_games;
  vector[num_teams] prior_score;
  array[num_games] int<lower=1> home_idx;
  array[num_games] int<lower=1> away_idx;
  array[num_games] int<lower=0> home_goals;
  array[num_games] int<lower=0> away_goals;
  array[num_games] int<lower=0, upper=1> not_neutral;
}
parameters {
  real mu0;
  real home_adv;
  real<lower=-0.1, upper=0.1> rho;     // bounded so the DC correction stays positive
  real b_att;
  real b_def;
  real<lower=0> sigma_att;
  real<lower=0> sigma_def;
  vector[num_teams] att_raw;
  vector[num_teams] def_raw;
}
transformed parameters {
  vector[num_teams] attack  = b_att * prior_score + sigma_att * att_raw;
  vector[num_teams] defence = b_def * prior_score + sigma_def * def_raw;
}
model {
  att_raw ~ std_normal();
  def_raw ~ std_normal();
  mu0 ~ normal(0, 1);
  home_adv ~ normal(0.25, 0.25);
  rho ~ normal(0, 0.05);
  b_att ~ normal(0, 1);
  b_def ~ normal(0, 1);
  sigma_att ~ normal(0, 1);
  sigma_def ~ normal(0, 1);
  for (n in 1:num_games) {
    real lh = exp(mu0 + home_adv * not_neutral[n] + attack[home_idx[n]] - defence[away_idx[n]]);
    real la = exp(mu0 + attack[away_idx[n]] - defence[home_idx[n]]);
    target += poisson_lpmf(home_goals[n] | lh);
    target += poisson_lpmf(away_goals[n] | la);
    target += dc_log_tau(home_goals[n], away_goals[n], lh, la, rho);
  }
}
"""

dc_data = {
    "num_teams":   len(teams),
    "num_games":   len(training),
    "prior_score": prior_score.tolist(),
    "home_idx":    training['home_idx'].astype(int).tolist(),
    "away_idx":    training['away_idx'].astype(int).tolist(),
    "home_goals":  training['home_goals'].astype(int).tolist(),
    "away_goals":  training['away_goals'].astype(int).tolist(),
    "not_neutral": training['not_neutral'].astype(int).tolist(),
}

print("Compiling Dixon-Coles model...")
dc_post = stan.build(dc_code, data=dc_data, random_seed=42)
print("Sampling (4 chains x 2000 draws)...")
dc_fit = dc_post.sample(num_chains=4, num_samples=2000, num_warmup=1000)
print("Done.")

att_s = np.asarray(dc_fit["attack"]);  def_s = np.asarray(dc_fit["defence"])
mu0_s = np.asarray(dc_fit["mu0"]).ravel()
ha_s  = np.asarray(dc_fit["home_adv"]).ravel()
rho_s = np.asarray(dc_fit["rho"]).ravel()
att_mean = {t: att_s[team_idx[t]-1].mean() for t in teams}
def_mean = {t: def_s[team_idx[t]-1].mean() for t in teams}
print(f"mu0={mu0_s.mean():.3f}  home_adv={ha_s.mean():.3f}  rho={rho_s.mean():.4f}")
print(f"b_att={np.asarray(dc_fit['b_att']).mean():.3f}  b_def={np.asarray(dc_fit['b_def']).mean():.3f}")

## 4. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18,4)); fig.patch.set_facecolor(BG)
fig.suptitle('Dixon-Coles posterior parameters', fontsize=13, fontweight='bold', color=DARK)
for ax, samp, label, col in [
    (axes[0], mu0_s, 'mu0 (baseline log-rate)', BLUE),
    (axes[1], ha_s,  'home_adv', GOLD),
    (axes[2], rho_s, 'rho (low-score dependence)', RED),
]:
    ax.set_facecolor(BG); ax.hist(samp, bins=50, color=col, alpha=0.8)
    ax.axvline(samp.mean(), color=DARK, ls='--', lw=2)
    ax.set_title(f"{label}\nmean={samp.mean():.3f}", fontsize=10, color=DARK)
    ax.spines[['top','right']].set_visible(False); ax.grid(alpha=0.3, ls=':')

ax = axes[3]; ax.set_facecolor(BG)
qx = [att_mean[t] for t in sorted(qualified)]; qy = [def_mean[t] for t in sorted(qualified)]
ax.scatter(qx, qy, s=18, color=BLUE, alpha=0.7)
for t in ["Argentina","Spain","Brazil","Germany","Haiti","Curaçao","New Zealand","Morocco","Mexico"]:
    ax.annotate(t, (att_mean[t], def_mean[t]), fontsize=7, color=DARK)
ax.set_xlabel('attack (higher = scores more)'); ax.set_ylabel('defence (higher = concedes less)')
ax.set_title('Attack vs defence (48 qualifiers)', fontsize=10, color=DARK)
ax.axhline(0, color=GRAY, ls=':', alpha=0.5); ax.axvline(0, color=GRAY, ls=':', alpha=0.5)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 5. Predictions — full scoreline distribution

For each fixture (treated as neutral, like the Student-t notebook) we build the joint scoreline
matrix per posterior draw, apply the $\tau$ correction, renormalise, and aggregate to
win/draw/loss. Because the model is over scorelines we also get the **most likely exact score**
for free — no ±0.5 threshold anywhere.

In [ ]:
def predict_match_dc(t1, t2, maxg=10, neutral=True):
    i, j = team_idx[t1]-1, team_idx[t2]-1
    boost = 0.0 if neutral else ha_s
    l1 = np.exp(mu0_s + boost + att_s[i] - def_s[j])   # team1 ("home")
    l2 = np.exp(mu0_s + att_s[j] - def_s[i])           # team2 ("away")
    xs = np.arange(maxg+1)
    p1 = poisson.pmf(xs[:,None], l1[None,:])           # (X, S)
    p2 = poisson.pmf(xs[:,None], l2[None,:])           # (Y, S)
    J = p1[:,None,:] * p2[None,:,:]                    # (X, Y, S)
    J[0,0] *= (1 - l1*l2*rho_s); J[1,0] *= (1 + l2*rho_s)
    J[0,1] *= (1 + l1*rho_s);    J[1,1] *= (1 - rho_s)
    J /= J.sum(axis=(0,1), keepdims=True)
    Xg = np.arange(maxg+1)[:,None]; Yg = np.arange(maxg+1)[None,:]
    w1 = (J * (Xg>Yg)[:,:,None]).sum((0,1)).mean()
    dr = (J * (Xg==Yg)[:,:,None]).sum((0,1)).mean()
    w2 = (J * (Xg<Yg)[:,:,None]).sum((0,1)).mean()
    mscore = np.unravel_index(J.mean(2).argmax(), (maxg+1, maxg+1))
    return w1, dr, w2, f"{mscore[0]}-{mscore[1]}"

fixtures = df_all[
    (df_all['tournament']=='FIFA World Cup') & (df_all['date']>='2026-06-01') &
    (df_all['home_team'].isin(qualified)) & (df_all['away_team'].isin(qualified))
][['date','home_team','away_team']].copy()

rows = []
for _, r in fixtures.iterrows():
    w1, d, w2, sc = predict_match_dc(r['home_team'], r['away_team'])
    rows.append({'group':team_group[r['home_team']], 'date':r['date'],
                 'team1':r['home_team'], 'team2':r['away_team'],
                 'team1_win':round(w1*100,1), 'draw':round(d*100,1),
                 'team2_win':round(w2*100,1), 'score':sc})
results_dc = pd.DataFrame(rows).sort_values(['group','date']).reset_index(drop=True)
print(f"{len(results_dc)} fixtures predicted. Sample:")
print(results_dc[['group','team1','team2','team1_win','draw','team2_win','score']].head(8).to_string(index=False))

## 6. Student-t vs Dixon-Coles comparison

We also fit the original Student-t score-difference model on the **same** training data and
compare. The headline question from caveat §9.1 is whether Dixon-Coles produces healthier draw
probabilities, especially for evenly-matched games.

In [ ]:
# --- Fit the Student-t model (same data, favourite/underdog ordering on score difference) ---
def build_training_t(matches, team_idx, rankings):
    rows=[]
    for _, r in matches.iterrows():
        h,a = r['home_team'], r['away_team']
        if rankings.get(h,999) <= rankings.get(a,999): fav,und,sd = h,a, r['home_score']-r['away_score']
        else:                                          fav,und,sd = a,h, r['away_score']-r['home_score']
        if fav in team_idx and und in team_idx:
            rows.append({'i':team_idx[fav],'j':team_idx[und],'sd':float(sd)})
    return pd.DataFrame(rows)

tr_t = build_training_t(matches, team_idx, rankings)
t_code = """
data { int<lower=0> num_teams; int<lower=0> num_games; vector[num_teams] prior_score;
  array[num_games] int team_1_idx; array[num_games] int team_2_idx; vector[num_games] score_diff; real<lower=0> deg_freedom; }
parameters { real b; real<lower=0> sigma_a; real<lower=0> sigma_y; vector[num_teams] raw_a; }
transformed parameters { vector[num_teams] a = b * prior_score + sigma_a * raw_a; }
model { raw_a ~ std_normal();
  for (i in 1:num_games) score_diff[i] ~ student_t(deg_freedom, a[team_1_idx[i]] - a[team_2_idx[i]], sigma_y); }
"""
t_data = {"num_teams":len(teams),"num_games":len(tr_t),"prior_score":prior_score.tolist(),
          "team_1_idx":tr_t['i'].astype(int).tolist(),"team_2_idx":tr_t['j'].astype(int).tolist(),
          "score_diff":tr_t['sd'].tolist(),"deg_freedom":7.0}
print("Compiling/sampling Student-t model (cached if already built)...")
t_fit = stan.build(t_code, data=t_data, random_seed=42).sample(num_chains=4, num_samples=2000, num_warmup=1000)
a_s = np.asarray(t_fit["a"]); sy_s = np.asarray(t_fit["sigma_y"]).ravel()

def predict_match_t(t1, t2):
    fav,und,ft = (t1,t2,True) if rankings.get(t1,999)<=rankings.get(t2,999) else (t2,t1,False)
    diff = a_s[team_idx[fav]-1] - a_s[team_idx[und]-1]
    yp = t_dist.rvs(df=7, loc=diff, scale=sy_s)
    fw=np.mean(yp>0.5); d=np.mean((yp>=-0.5)&(yp<=0.5)); uw=np.mean(yp<-0.5)
    return (fw,d,uw) if ft else (uw,d,fw)

comp = results_dc[['group','team1','team2','team1_win','draw','team2_win']].copy()
comp.columns = ['group','team1','team2','t1_dc','draw_dc','t2_dc']
tt = [predict_match_t(r['team1'], r['team2']) for _,r in comp.iterrows()]
comp['draw_t'] = [round(x[1]*100,1) for x in tt]
comp['t1_t']   = [round(x[0]*100,1) for x in tt]
print(f"Mean draw probability  —  Dixon-Coles: {comp['draw_dc'].mean():.1f}%   Student-t: {comp['draw_t'].mean():.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(14,5.5)); fig.patch.set_facecolor(BG)
for a_ in ax: a_.set_facecolor(BG)
ax[0].scatter(comp['draw_t'], comp['draw_dc'], s=22, color=BLUE, alpha=0.7)
lo,hi = 5, max(comp['draw_t'].max(), comp['draw_dc'].max())+2
ax[0].plot([lo,hi],[lo,hi], color=GRAY, ls='--'); ax[0].set_xlim(lo,hi); ax[0].set_ylim(lo,hi)
ax[0].set_xlabel('Draw % (Student-t, ±0.5 threshold)'); ax[0].set_ylabel('Draw % (Dixon-Coles)')
ax[0].set_title('Draw probability per fixture', fontsize=11, color=DARK)
ax[0].spines[['top','right']].set_visible(False); ax[0].grid(alpha=0.3, ls=':')
ax[1].scatter(comp['t1_t'], comp['t1_dc'], s=22, color=RED, alpha=0.7)
ax[1].plot([0,100],[0,100], color=GRAY, ls='--'); ax[1].set_xlim(0,100); ax[1].set_ylim(0,100)
ax[1].set_xlabel('Team-1 win % (Student-t)'); ax[1].set_ylabel('Team-1 win % (Dixon-Coles)')
ax[1].set_title('Team-1 win probability per fixture', fontsize=11, color=DARK)
ax[1].spines[['top','right']].set_visible(False); ax[1].grid(alpha=0.3, ls=':')
plt.tight_layout(); plt.show()

comp['draw_diff'] = (comp['draw_dc'] - comp['draw_t']).round(1)
print("\nLargest draw-probability differences (DC − Student-t):")
print(comp.reindex(comp['draw_diff'].abs().sort_values(ascending=False).index)
      [['team1','team2','draw_t','draw_dc','draw_diff']].head(8).to_string(index=False))

## 7. Full fixture list — official kickoff times, Dixon-Coles predictions + most-likely score

In [ ]:
official_schedule = [   # (team1, team2, "YYYY-MM-DD HH:MM" ET, "venue, city")
    ("Mexico","South Africa","2026-06-11 15:00","Estadio Azteca, Mexico City"),
    ("South Korea","Czech Republic","2026-06-11 22:00","Estadio Akron, Zapopan"),
    ("Canada","Bosnia and Herzegovina","2026-06-12 15:00","BMO Field, Toronto"),
    ("United States","Paraguay","2026-06-12 21:00","SoFi Stadium, Inglewood"),
    ("Qatar","Switzerland","2026-06-13 15:00","Levi's Stadium, Santa Clara"),
    ("Brazil","Morocco","2026-06-13 18:00","MetLife Stadium, East Rutherford"),
    ("Haiti","Scotland","2026-06-13 21:00","Gillette Stadium, Foxborough"),
    ("Australia","Turkey","2026-06-14 00:00","BC Place, Vancouver"),
    ("Germany","Curaçao","2026-06-14 13:00","NRG Stadium, Houston"),
    ("Netherlands","Japan","2026-06-14 16:00","AT&T Stadium, Arlington"),
    ("Ivory Coast","Ecuador","2026-06-14 19:00","Lincoln Financial Field, Philadelphia"),
    ("Sweden","Tunisia","2026-06-14 22:00","Estadio BBVA, Monterrey"),
    ("Spain","Cape Verde","2026-06-15 12:00","Mercedes-Benz Stadium, Atlanta"),
    ("Belgium","Egypt","2026-06-15 15:00","Lumen Field, Seattle"),
    ("Saudi Arabia","Uruguay","2026-06-15 18:00","Hard Rock Stadium, Miami Gardens"),
    ("Iran","New Zealand","2026-06-15 21:00","SoFi Stadium, Inglewood"),
    ("France","Senegal","2026-06-16 15:00","MetLife Stadium, East Rutherford"),
    ("Iraq","Norway","2026-06-16 18:00","Gillette Stadium, Foxborough"),
    ("Argentina","Algeria","2026-06-16 21:00","Arrowhead Stadium, Kansas City"),
    ("Austria","Jordan","2026-06-17 00:00","Levi's Stadium, Santa Clara"),
    ("Portugal","DR Congo","2026-06-17 13:00","NRG Stadium, Houston"),
    ("England","Croatia","2026-06-17 16:00","AT&T Stadium, Arlington"),
    ("Ghana","Panama","2026-06-17 19:00","BMO Field, Toronto"),
    ("Uzbekistan","Colombia","2026-06-17 22:00","Estadio Azteca, Mexico City"),
    ("Czech Republic","South Africa","2026-06-18 12:00","Mercedes-Benz Stadium, Atlanta"),
    ("Switzerland","Bosnia and Herzegovina","2026-06-18 15:00","SoFi Stadium, Inglewood"),
    ("Canada","Qatar","2026-06-18 18:00","BC Place, Vancouver"),
    ("Mexico","South Korea","2026-06-18 21:00","Estadio Akron, Zapopan"),
    ("United States","Australia","2026-06-19 15:00","Lumen Field, Seattle"),
    ("Scotland","Morocco","2026-06-19 18:00","Gillette Stadium, Foxborough"),
    ("Brazil","Haiti","2026-06-19 20:30","Lincoln Financial Field, Philadelphia"),
    ("Turkey","Paraguay","2026-06-19 23:00","Levi's Stadium, Santa Clara"),
    ("Netherlands","Sweden","2026-06-20 13:00","NRG Stadium, Houston"),
    ("Germany","Ivory Coast","2026-06-20 16:00","BMO Field, Toronto"),
    ("Ecuador","Curaçao","2026-06-20 20:00","Arrowhead Stadium, Kansas City"),
    ("Tunisia","Japan","2026-06-21 00:00","Estadio BBVA, Monterrey"),
    ("Spain","Saudi Arabia","2026-06-21 12:00","Mercedes-Benz Stadium, Atlanta"),
    ("Belgium","Iran","2026-06-21 15:00","SoFi Stadium, Inglewood"),
    ("Uruguay","Cape Verde","2026-06-21 18:00","Hard Rock Stadium, Miami Gardens"),
    ("New Zealand","Egypt","2026-06-21 21:00","BC Place, Vancouver"),
    ("Argentina","Austria","2026-06-22 13:00","AT&T Stadium, Arlington"),
    ("France","Iraq","2026-06-22 17:00","Lincoln Financial Field, Philadelphia"),
    ("Norway","Senegal","2026-06-22 20:00","MetLife Stadium, East Rutherford"),
    ("Jordan","Algeria","2026-06-22 23:00","Levi's Stadium, Santa Clara"),
    ("Portugal","Uzbekistan","2026-06-23 13:00","NRG Stadium, Houston"),
    ("England","Ghana","2026-06-23 16:00","Gillette Stadium, Foxborough"),
    ("Panama","Croatia","2026-06-23 19:00","BMO Field, Toronto"),
    ("Colombia","DR Congo","2026-06-23 22:00","Estadio Akron, Zapopan"),
    ("Switzerland","Canada","2026-06-24 15:00","BC Place, Vancouver"),
    ("Bosnia and Herzegovina","Qatar","2026-06-24 15:00","Lumen Field, Seattle"),
    ("Scotland","Brazil","2026-06-24 18:00","Hard Rock Stadium, Miami Gardens"),
    ("Morocco","Haiti","2026-06-24 18:00","Mercedes-Benz Stadium, Atlanta"),
    ("Czech Republic","Mexico","2026-06-24 21:00","Estadio Azteca, Mexico City"),
    ("South Africa","South Korea","2026-06-24 21:00","Estadio BBVA, Monterrey"),
    ("Curaçao","Ivory Coast","2026-06-25 16:00","Lincoln Financial Field, Philadelphia"),
    ("Ecuador","Germany","2026-06-25 16:00","MetLife Stadium, East Rutherford"),
    ("Japan","Sweden","2026-06-25 19:00","AT&T Stadium, Arlington"),
    ("Tunisia","Netherlands","2026-06-25 19:00","Arrowhead Stadium, Kansas City"),
    ("Turkey","United States","2026-06-25 22:00","SoFi Stadium, Inglewood"),
    ("Paraguay","Australia","2026-06-25 22:00","Levi's Stadium, Santa Clara"),
    ("Norway","France","2026-06-26 15:00","Gillette Stadium, Foxborough"),
    ("Senegal","Iraq","2026-06-26 15:00","BMO Field, Toronto"),
    ("Cape Verde","Saudi Arabia","2026-06-26 20:00","NRG Stadium, Houston"),
    ("Uruguay","Spain","2026-06-26 20:00","Estadio Akron, Zapopan"),
    ("Egypt","Iran","2026-06-26 23:00","Lumen Field, Seattle"),
    ("New Zealand","Belgium","2026-06-26 23:00","BC Place, Vancouver"),
    ("Panama","England","2026-06-27 17:00","MetLife Stadium, East Rutherford"),
    ("Croatia","Ghana","2026-06-27 17:00","Lincoln Financial Field, Philadelphia"),
    ("Colombia","Portugal","2026-06-27 19:30","Hard Rock Stadium, Miami Gardens"),
    ("DR Congo","Uzbekistan","2026-06-27 19:30","Mercedes-Benz Stadium, Atlanta"),
    ("Algeria","Austria","2026-06-27 22:00","Arrowhead Stadium, Kansas City"),
    ("Jordan","Argentina","2026-06-27 22:00","AT&T Stadium, Arlington"),
]
kickoff = {frozenset((a,b)): (ts,v) for a,b,ts,v in official_schedule}
assert len(kickoff)==72, "expected 72 unique fixtures"

s = results_dc.copy()
keys = [frozenset((r.team1, r.team2)) for r in s.itertuples()]
assert all(k in kickoff for k in keys), "fixture/kickoff mismatch"
s['kickoff_et'] = pd.to_datetime([kickoff[k][0] for k in keys])
s['venue']      = [kickoff[k][1] for k in keys]
s = s.sort_values(['kickoff_et','group']).reset_index(drop=True)

table = pd.DataFrame({
    'Date':         s['kickoff_et'].dt.strftime('%a %b %d'),
    'Kickoff (ET)': s['kickoff_et'].dt.strftime('%I:%M %p').str.lstrip('0'),
    'Grp':          s['group'],
    'Match (Team1 v Team2)': s['team1'] + '  v  ' + s['team2'],
    'T1 win %':     s['team1_win'], 'Draw %': s['draw'], 'T2 win %': s['team2_win'],
    'Likely score': s['score'], 'Venue': s['venue'],
})
table.index = range(1, len(table)+1); table.index.name = 'Game'
print(f"{len(table)} group-stage fixtures — Dixon-Coles, official kickoff order (ET):\n")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None, 'display.width', 220):
    display(table)

## 8. Notes & caveats

- **Draws are now intrinsic.** With negative `rho` the model lifts low-score draws (0-0, 1-1)
  without any threshold, and for evenly-matched fixtures the draw probability is higher than the
  Student-t version — see §6.
- **Home advantage** is estimated (`home_adv`) and applied to non-neutral training games; the
  2026 fixtures are predicted as neutral, matching the Student-t notebook. Set `neutral=False` in
  `predict_match_dc` to give a side home advantage.
- **`rho` is hard-bounded** to ±0.1 to keep the correction positive; it lands well inside that range.
- **Still no temporal decay.** Dixon-Coles' original paper down-weights older games — a natural
  next step (add a per-game weight to the `target +=` terms).
- The five prior-only qualifiers (Algeria, Egypt, Ivory Coast, New Zealand, Panama) are unchanged:
  their attack/defence are driven by the FIFA-ranking prior alone.